<a href="https://colab.research.google.com/github/shalomalena/Crop-Management-Simulation/blob/main/SharedTask1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision torchaudio
!pip install nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [3]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize
from tqdm import tqdm
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
train_path ='/content/drive/MyDrive/Research/subtask1/train.json'
val_path ='/content/drive/MyDrive/Research/subtask1/validation.json'

In [10]:
with open(train_path, 'r') as f:
    train_data = json.load(f)

with open(val_path, 'r') as f:
    val_data = json.load(f)

print(json.dumps(train_data[:2], indent=4))



[
    {
        "id": "65635",
        "text": "THIS IS WHY YOU NEED\\n\\nA SHARPIE WITH YOU AT ALL TIMES",
        "labels": [
            "Black-and-white Fallacy/Dictatorship"
        ],
        "link": "https://www.facebook.com/photo/?fbid=4023552137722493&set=g.633131750534436"
    },
    {
        "id": "67927",
        "text": "GOOD NEWS!\\n\\nNAZANIN ZAGHARI-RATCLIFFE AND ANOOSHEH ASHOORI HAVE BEEN RELEASED\\n\\nAfter years of being unjustly detained in Iran, they are making their way safely back to the UK.",
        "labels": [
            "Loaded Language",
            "Glittering generalities (Virtue)"
        ],
        "link": "https://www.facebook.com/amnesty/photos/5311988665480629/"
    }
]


In [1]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2025-03-24 17:17:55--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-03-24 17:17:55--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-03-24 17:17:56--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [6]:
def embeddings(glove_file_path):
  embedding = {}
  with open(glove_file_path, 'r', encoding='utf-8') as f:
    for line in f:
      values = line.split()
      word = values[0]
      vector = np.array(values[1:], dtype='float32')
      embedding[word] = vector
  return embedding

glove_path = '/content/glove.6B.50d.txt'
glove_embeddings = embeddings(glove_path)

embedding_dim = 50

In [7]:
label_list = [
    "presenting irrelevant data (red herring)",
    "misrepresentation of someone's position (straw man)",
    "whataboutism",
    "causal oversimplification",
    "obfuscation, intentional vagueness, confusion",
    "appeal to authority",
    "black-and-white fallacy/dictatorship",
    "name calling/labeling",
    "loaded language",
    "exaggeration/minimisation",
    "flag-waving",
    "doubt",
    "appeal to fear/prejudice",
    "slogans",
    "thought-terminating cliché",
    "bandwagon",
    "reductio ad hitlerum",
    "repetition",
    "smears",
    "glittering generalities (virtue)"
]



In [8]:
from torch.utils.data import Dataset
import numpy as np
import torch
from nltk.tokenize import word_tokenize

class MemeDataset(Dataset):
    def __init__(self, data, embeddings, label_list):
        self.data = data
        self.embeddings = embeddings
        self.label_list = label_list

    def text_to_vec(self, text):
        tokens = word_tokenize(text.lower())
        vectors = []

        for token in tokens:
            if token in self.embeddings:
                vectors.append(self.embeddings[token])


        if len(vectors) == 0:
            return np.zeros(embedding_dim)

        return np.mean(vectors, axis=0)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        meme = self.data[idx]
        text_vector = self.text_to_vec(meme['text'])

        label_vector = np.zeros(len(self.label_list))

        for label in meme['labels']:
            label = label.lower()
            if label not in self.label_list:
                print(f"Label '{label}' not found in label_list!")
                continue
            else:
                label_vector[self.label_list.index(label)] = 1.0

        return torch.tensor(text_vector, dtype=torch.float32), torch.tensor(label_vector, dtype=torch.float32)


In [11]:
train_datset = MemeDataset(train_data, glove_embeddings, label_list)
val_dataset = MemeDataset(val_data, glove_embeddings, label_list)

train_loader = DataLoader(train_datset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [12]:
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(MLP, self).__init__()
    self.fc1 = nn.Linear(input_dim, hidden_dim)
    self.fc2 = nn.Linear(hidden_dim, output_dim)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x


In [13]:
input_dim = 50
hidden_dim = 128
output_dim = len(label_list)


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim

model = MLP(input_dim, hidden_dim, output_dim)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [15]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


100%|██████████| 438/438 [00:03<00:00, 110.22it/s]


Epoch 1/10, Loss: 0.2731


100%|██████████| 438/438 [00:07<00:00, 55.28it/s] 


Epoch 2/10, Loss: 0.2310


100%|██████████| 438/438 [00:03<00:00, 113.87it/s]


...

Epoch 8/10, Loss: 0.2156


100%|██████████| 438/438 [00:02<00:00, 167.76it/s]


Epoch 9/10, Loss: 0.2147


100%|██████████| 438/438 [00:02<00:00, 169.15it/s]

Epoch 10/10, Loss: 0.2139


In [16]:
def evaluate(model, loader):
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            val_loss += loss.item()

        avg_val_loss = val_loss / len(loader)
        return avg_val_loss

    return val_data / len(dataloader)
val_loss = evaluate(model, val_loader)
print(f"Validation Loss after epoch {epoch+1}: {val_loss}")

Validation Loss after epoch 10: 0.2278614016249776
